# Debug Loading of Event Numbers

In [7]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import logging
import awkward as ak
import uproot

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch
from pyedm4hep.utils import ecal, hcal, all_trackers, all_calos

from pathlib import Path
import logging

# Reuse postprocessing helpers
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")
from convert_particles import build_particles_df_with_parents_and_vertex, write_particles_with_selection
from convert_digihits import process_event_for_digihits, write_digihits_with_selection
from utils.path_utils import get_run_paths, make_dir
from utils.track_utils import load_root_file, load_track_summary, create_particle_barcode_map

from convert_all import convert_all


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
run_dir = Path("/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0")

In [28]:
event_idx = 5
edm4hep_path = run_dir / "edm4hep.root"
run_size = 128
batch = EDM4hepEvent(str(edm4hep_path), event_index=event_idx)

Augmenting particles
Ready with tracker hits Index(['cellID', 'time', 'x', 'y', 'z', 'particle_id', 'detector', 'r', 'R',
       'phi', 'theta', 'eta'],
      dtype='object')
Augmenting particle hit counts with tracker hits


EDM4Hep Loading

In [29]:
edm_hits_all = batch.get_tracker_hits_df()

In [30]:
edm_hits_all

,cellID,time,x,y,z,particle_id,detector,r,R,phi,theta,eta
0,263608214945814,-0.988947,68.107020,0.662978,-266.737257,91640,PixelBarrelReadout,68.110247,275.295786,0.009734,2.891589,-2.074199
1,269449622126870,-1.003263,66.425640,14.680769,-262.485360,91639,PixelBarrelReadout,68.028601,271.157620,0.217514,2.888001,-2.059799
2,247048817345574,-0.201598,97.351430,58.516660,-496.313792,91639,PixelBarrelReadout,113.584772,509.145245,0.541218,2.916610,-2.180650
3,234881325991206,-0.171200,97.753035,60.592826,-505.209035,91639,PixelBarrelReadout,115.009332,518.134457,0.554892,2.917760,-2.185817
4,246430375609382,-0.199948,97.347790,58.632083,-496.755771,91646,PixelBarrelReadout,113.641161,509.588667,0.542105,2.916696,-2.181034
...,...,...,...,...,...,...,...,...,...,...,...,...
260241,17532057690382,2483.650635,-262.052827,-968.990803,1320.597476,944106,LongStripEndcapReadout,1003.800209,1658.792499,-1.834917,0.649941,1.087926
260242,16252157358366,19.697929,-449.969138,960.220820,1609.495851,949064,LongStripEndcapReadout,1060.422675,1927.426559,2.009018,0.582574,1.204589
260243,16806207066142,321.726868,299.712875,793.902061,1574.613356,949548,LongStripEndcapReadout,848.591946,1788.718958,1.209819,0.494297,1.377109
260244,16810502033438,321.730164,300.082792,794.201939,1574.625000,949548,LongStripEndcapReadout,849.003182,1788.924340,1.209536,0.494497,1.376689


Direct root loading

In [8]:
root_file = uproot.open(edm4hep_path)

In [15]:
event_tree = root_file["events"]


In [18]:
event_tree["EventHeader/EventHeader.eventNumber"].arrays()

<Array [{...}, {...}, {...}, ..., {...}, {...}] type='128 * {"EventHeader.e...'>

Measurements file

In [39]:
uproot.open(measurements_path)["measurements"].keys()

['event_nr',
 'volume_id',
 'layer_id',
 'surface_id',
 'extra_id',
 'rec_loc0',
 'rec_loc1',
 'rec_time',
 'var_loc0',
 'var_loc1',
 'var_time',
 'rec_x',
 'rec_y',
 'rec_z',
 'clus_size',
 'channel_value',
 'channel_loc0',
 'clus_size_loc0',
 'channel_loc1',
 'clus_size_loc1',
 'true_loc0',
 'true_loc1',
 'true_phi',
 'true_theta',
 'true_qop',
 'true_time',
 'true_x',
 'true_y',
 'true_z',
 'true_incident_phi',
 'true_incident_theta',
 'residual_loc0',
 'residual_loc1',
 'residual_time',
 'pull_loc0',
 'pull_loc1',
 'pull_time']

In [35]:
measurements_path = run_dir / "measurements.root"
included_columns = [
    "event_nr",
    "true_x",
    "true_y",
    "true_z",
    "particle_id",
]
measurements_df_all = load_root_file(str(measurements_path), included_columns=included_columns, event_id=event_idx)
measurements_df_all


,event_nr,true_x,true_y,true_z
entry,,,,
0,5,62.670483,7.816412,-1515.599976
1,5,47.837719,5.069315,-1515.599976
2,5,67.374641,8.194266,-1515.599976
3,5,79.020744,4.660388,-1515.599976
4,5,54.200138,1.408986,-1515.599976
...,...,...,...,...
260232,5,315.218903,891.565735,3009.500000
260233,5,353.883423,939.871887,3009.500000
260234,5,371.864777,856.458740,3009.500000


In [41]:
measurements_df_all.drop_duplicates(subset=["true_x", "true_y", "true_z"])

,event_nr,true_x,true_y,true_z
entry,,,,
0,5,62.670483,7.816412,-1515.599976
1,5,47.837719,5.069315,-1515.599976
2,5,67.374641,8.194266,-1515.599976
3,5,79.020744,4.660388,-1515.599976
4,5,54.200138,1.408986,-1515.599976
...,...,...,...,...
260232,5,315.218903,891.565735,3009.500000
260233,5,353.883423,939.871887,3009.500000
260234,5,371.864777,856.458740,3009.500000


In [34]:
# Convert all x, y, z, true_x, true_y, true_z to float32
edm_hits_all["x"] = edm_hits_all["x"].astype(np.float32)
edm_hits_all["y"] = edm_hits_all["y"].astype(np.float32)
edm_hits_all["z"] = edm_hits_all["z"].astype(np.float32)
measurements_df_all["true_x"] = measurements_df_all["true_x"].astype(np.float32)
measurements_df_all["true_y"] = measurements_df_all["true_y"].astype(np.float32)
measurements_df_all["true_z"] = measurements_df_all["true_z"].astype(np.float32)

# merge measurements and edm4hep
merged_df = edm_hits_all.merge(measurements_df_all, left_on=["x", "y", "z"], right_on=["true_x", "true_y", "true_z"], how="inner")
merged_df


,cellID,time,x,y,z,particle_id,detector,r,R,phi,theta,eta,event_nr,true_x,true_y,true_z
0,263608214945814,-0.988947,68.107018,0.662978,-266.737244,91640,PixelBarrelReadout,68.110247,275.295786,0.009734,2.891589,-2.074199,5,68.107018,0.662978,-266.737244
1,269449622126870,-1.003263,66.425636,14.680769,-262.485352,91639,PixelBarrelReadout,68.028601,271.157620,0.217514,2.888001,-2.059799,5,66.425636,14.680769,-262.485352
2,247048817345574,-0.201598,97.351433,58.516659,-496.313782,91639,PixelBarrelReadout,113.584772,509.145245,0.541218,2.916610,-2.180650,5,97.351433,58.516659,-496.313782
3,234881325991206,-0.171200,97.753036,60.592827,-505.209045,91639,PixelBarrelReadout,115.009332,518.134457,0.554892,2.917760,-2.185817,5,97.753036,60.592827,-505.209045
4,246430375609382,-0.199948,97.347794,58.632084,-496.755768,91646,PixelBarrelReadout,113.641161,509.588667,0.542105,2.916696,-2.181034,5,97.347794,58.632084,-496.755768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260240,17532057690382,2483.650635,-262.052826,-968.990784,1320.597534,944106,LongStripEndcapReadout,1003.800209,1658.792499,-1.834917,0.649941,1.087926,5,-262.052826,-968.990784,1320.597534
260241,16252157358366,19.697929,-449.969147,960.220825,1609.495850,949064,LongStripEndcapReadout,1060.422675,1927.426559,2.009018,0.582574,1.204589,5,-449.969147,960.220825,1609.495850
260242,16806207066142,321.726868,299.712860,793.902039,1574.613403,949548,LongStripEndcapReadout,848.591946,1788.718958,1.209819,0.494297,1.377109,5,299.712860,793.902039,1574.613403
260243,16810502033438,321.730164,300.082794,794.201965,1574.625000,949548,LongStripEndcapReadout,849.003182,1788.924340,1.209536,0.494497,1.376689,5,300.082794,794.201965,1574.625000


In [49]:
merged_df.drop_duplicates(subset=["x", "y", "z", "particle_id"])

,cellID,time,x,y,z,particle_id,detector,r,R,phi,theta,eta,event_nr,true_x,true_y,true_z
0,263608214945814,-0.988947,68.107018,0.662978,-266.737244,91640,PixelBarrelReadout,68.110247,275.295786,0.009734,2.891589,-2.074199,5,68.107018,0.662978,-266.737244
1,269449622126870,-1.003263,66.425636,14.680769,-262.485352,91639,PixelBarrelReadout,68.028601,271.157620,0.217514,2.888001,-2.059799,5,66.425636,14.680769,-262.485352
2,247048817345574,-0.201598,97.351433,58.516659,-496.313782,91639,PixelBarrelReadout,113.584772,509.145245,0.541218,2.916610,-2.180650,5,97.351433,58.516659,-496.313782
3,234881325991206,-0.171200,97.753036,60.592827,-505.209045,91639,PixelBarrelReadout,115.009332,518.134457,0.554892,2.917760,-2.185817,5,97.753036,60.592827,-505.209045
4,246430375609382,-0.199948,97.347794,58.632084,-496.755768,91646,PixelBarrelReadout,113.641161,509.588667,0.542105,2.916696,-2.181034,5,97.347794,58.632084,-496.755768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260240,17532057690382,2483.650635,-262.052826,-968.990784,1320.597534,944106,LongStripEndcapReadout,1003.800209,1658.792499,-1.834917,0.649941,1.087926,5,-262.052826,-968.990784,1320.597534
260241,16252157358366,19.697929,-449.969147,960.220825,1609.495850,949064,LongStripEndcapReadout,1060.422675,1927.426559,2.009018,0.582574,1.204589,5,-449.969147,960.220825,1609.495850
260242,16806207066142,321.726868,299.712860,793.902039,1574.613403,949548,LongStripEndcapReadout,848.591946,1788.718958,1.209819,0.494297,1.377109,5,299.712860,793.902039,1574.613403
260243,16810502033438,321.730164,300.082794,794.201965,1574.625000,949548,LongStripEndcapReadout,849.003182,1788.924340,1.209536,0.494497,1.376689,5,300.082794,794.201965,1574.625000


In [46]:
measurements_df_all.merge(edm_hits_all[edm_hits_all.duplicated(subset=["x", "y", "z"], keep=False)], left_on=["true_x", "true_y", "true_z"], right_on=["x", "y", "z"], how="inner")

,event_nr,true_x,true_y,true_z,cellID,time,x,y,z,particle_id,detector,r,R,phi,theta,eta
0,5,-162.132065,-254.539749,-2192.539551,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397894,ShortStripEndcapReadout,301.790146,2213.211964,-2.137951,3.004808,-2.680934
1,5,-162.132065,-254.539749,-2192.539551,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397893,ShortStripEndcapReadout,301.790149,2213.211982,-2.137951,3.004808,-2.680934
2,5,-162.132065,-254.539749,-2192.539551,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397894,ShortStripEndcapReadout,301.790146,2213.211964,-2.137951,3.004808,-2.680934
3,5,-162.132065,-254.539749,-2192.539551,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397893,ShortStripEndcapReadout,301.790149,2213.211982,-2.137951,3.004808,-2.680934
4,5,-140.190979,433.484985,-1552.622437,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408392,ShortStripEndcapReadout,455.590544,1618.085034,1.883584,2.856172,-1.940116
5,5,-140.190979,433.484985,-1552.622437,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408391,ShortStripEndcapReadout,455.590551,1618.085036,1.883584,2.856172,-1.940116
6,5,-140.190979,433.484985,-1552.622437,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408392,ShortStripEndcapReadout,455.590544,1618.085034,1.883584,2.856172,-1.940116
7,5,-140.190979,433.484985,-1552.622437,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408391,ShortStripEndcapReadout,455.590551,1618.085036,1.883584,2.856172,-1.940116
8,5,402.212158,-899.917175,-1925.603638,16810502156589,3.840398,402.212158,-899.917175,-1925.603638,794616,LongStripEndcapReadout,985.710669,2163.232528,-1.150487,2.668473,-1.422653
9,5,402.212158,-899.917175,-1925.603638,16810502156589,3.840398,402.212158,-899.917175,-1925.603638,794615,LongStripEndcapReadout,985.710659,2163.232523,-1.150487,2.668473,-1.422653


In [48]:
measurements_df_all[measurements_df_all.duplicated(subset=["true_x", "true_y", "true_z"], keep=False)]

,event_nr,true_x,true_y,true_z
entry,,,,
114394,5,-162.132065,-254.539749,-2192.539551
114395,5,-162.132065,-254.539749,-2192.539551
122881,5,-140.190979,433.484985,-1552.622437
122885,5,-140.190979,433.484985,-1552.622437
217556,5,402.212158,-899.917175,-1925.603638
217558,5,402.212158,-899.917175,-1925.603638
247344,5,305.504211,-897.876404,1325.620850
247345,5,305.504211,-897.876404,1325.620850


In [ ]:
edm_hits_all[edm_hits_all.duplicated(subset=["x", "y", "z"], keep=False)]

,cellID,time,x,y,z,particle_id,detector,r,R,phi,theta,eta
167460,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397894,ShortStripEndcapReadout,301.790146,2213.211964,-2.137951,3.004808,-2.680934
167461,16840566796346,16.239807,-162.132065,-254.539749,-2192.539551,397893,ShortStripEndcapReadout,301.790149,2213.211982,-2.137951,3.004808,-2.680934
167910,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408392,ShortStripEndcapReadout,455.590544,1618.085034,1.883584,2.856172,-1.940116
167913,356482298138,2.142216,-140.190979,433.484985,-1552.622437,408391,ShortStripEndcapReadout,455.590551,1618.085036,1.883584,2.856172,-1.940116
243460,16810502156589,3.840398,402.212158,-899.917175,-1925.603638,794616,LongStripEndcapReadout,985.710669,2163.232528,-1.150487,2.668473,-1.422653
243463,16810502156589,3.840398,402.212158,-899.917175,-1925.603638,794615,LongStripEndcapReadout,985.710659,2163.232523,-1.150487,2.668473,-1.422653
257693,1821066289422,0.691215,305.504211,-897.876404,1325.620850,759665,LongStripEndcapReadout,948.427543,1629.964839,-1.242832,0.621026,1.136647
257694,1821066289422,0.691215,305.504211,-897.876404,1325.620850,759664,LongStripEndcapReadout,948.427556,1629.964846,-1.242832,0.621026,1.136647
